# Генерируем и грузим синтетику по продажам и остаткам 

In [2]:
import os 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sqlalchemy import create_engine, text, inspect
from dotenv import load_dotenv
from pathlib import Path

# Определяем корень проекта 
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent 
print(NOTEBOOK_DIR)
print(PROJECT_ROOT)

# Путь к .env
env_path = PROJECT_ROOT / ".env"
print(f"📁 Ищем файл .env по пути: {env_path}")

if not env_path.exists():
    raise FileNotFoundError(
        f"❌ Файл .env не найден!\n"
        f"Убедитесь, что он лежит в корне проекта: {PROJECT_ROOT}"
    )

# Загружаем переменные окружения
load_dotenv(env_path)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# Создаем engine
engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Тест подключения
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
    
print("\n✅ Подключение к PostgreSQL успешно установлено!")


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


C:\Users\User\Desktop\Учеба\ПРОЕКТ_ПРОГНОЗ_ВОЗНИКНОВЕНИЯ_ДЕФЕЦИТА_ЛК_СРЕДСТВ\Проект\drug-deficit-predictor\notebooks
C:\Users\User\Desktop\Учеба\ПРОЕКТ_ПРОГНОЗ_ВОЗНИКНОВЕНИЯ_ДЕФЕЦИТА_ЛК_СРЕДСТВ\Проект\drug-deficit-predictor
📁 Ищем файл .env по пути: C:\Users\User\Desktop\Учеба\ПРОЕКТ_ПРОГНОЗ_ВОЗНИКНОВЕНИЯ_ДЕФЕЦИТА_ЛК_СРЕДСТВ\Проект\drug-deficit-predictor\.env

✅ Подключение к PostgreSQL успешно установлено!


In [4]:
# --- 2. Загрузка checkpoint ---
checkpoint_path = f'{NOTEBOOK_DIR}/checkpoint/df_aggregated_v3_clean.parquet'
output_dir = f'{NOTEBOOK_DIR}/checkpoint/'

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint не найден: {checkpoint_path}")

df_agg = pd.read_parquet(checkpoint_path)
print(f"✅ Загружено: {len(df_agg):,} строк × {len(df_agg.columns)} колонок")

✅ Загружено: 7,672 строк × 35 колонок


In [4]:
with pd.option_context('display.max_columns', None,
                    #    'display.max_rows', None,
                    #    'display.width', None,
                    #    'display.max_colwidth', None
                       ):
    print(df_agg)

                                      smnn_code  \
0     21.10.51.121-000001-1-00110-0000000000000   
1     21.10.51.121-000001-1-00120-0000000000000   
2     21.10.51.121-000001-1-00125-0000000000000   
3     21.10.51.121-000003-1-00166-0000000000000   
4     21.10.51.121-000003-1-00167-0000000000000   
...                                         ...   
7667  21.20.23.199-000012-1-00010-0000000000000   
7668  21.20.23.199-000036-1-00009-0000000000000   
7669  21.20.23.199-000043-1-00005-0000000000000   
7670  21.20.23.199-000044-1-00003-0000000000000   
7671  21.20.23.199-000046-1-00001-0000000000000   

                                           standard_inn  products_count  \
0                         ПОЛИВИТАМИНЫ+ПРОЧИЕ ПРЕПАРАТЫ              23   
1                         ПОЛИВИТАМИНЫ+ПРОЧИЕ ПРЕПАРАТЫ               2   
2                         ПОЛИВИТАМИНЫ+ПРОЧИЕ ПРЕПАРАТЫ             100   
3                                 ПОЛИВИТАМИНЫ+МИНЕРАЛЫ             243   
4           

In [5]:
# Популярность = сумма логарифмов счётчиков.
# log1p — сглаживает длинный хвост (1, 2, 100, 500...).
# Используется только для генерации продаж (не в X модели).

df = df_agg.copy()
df['popularity_score'] = (
    np.log1p(df['products_count']) +
    np.log1p(df['producers_count']) +
    np.log1p(df['trade_count'])
)
print(f"📊 popularity_score: mean={df['popularity_score'].mean():.2f}")

📊 popularity_score: mean=5.19


In [6]:
# risk_scaled — признаки в диапазоне [0, 1] (MinMax).
# risk_propensity — взвешенная сумма (все веса согласованы).
# target_class — по процентилям (60% низкий, 25% средний, 15% высокий). 
# Это «идеальный» риск из статики без шума. 
# Используется только для генерации продаж/остатков и для исключений 
# в новом target.

risk_cols = [
    'producers_count', 
    'products_count', 
    'has_russian_producer',
    'is_essential', 
    'is_narcotic', 
    'has_atc_substitutes'
]

scaler = MinMaxScaler()
risk_scaled = pd.DataFrame(scaler.fit_transform(df[risk_cols].astype(float)), columns=risk_cols, index=df.index)

df['risk_propensity'] = (
    + (1 - risk_scaled['producers_count']) * 0.30
    + (1 - risk_scaled['products_count']) * 0.20
    + (1 - risk_scaled['has_atc_substitutes']) * 0.15
    + risk_scaled['is_narcotic'] * 0.10
    - risk_scaled['is_essential'] * 0.15
    - risk_scaled['has_russian_producer'] * 0.10
)

df['risk_propensity'] = (
    (df['risk_propensity'] - df['risk_propensity'].min()) /
    (df['risk_propensity'].max() - df['risk_propensity'].min())
)

q60 = df['risk_propensity'].quantile(0.60)
q85 = df['risk_propensity'].quantile(0.85)

df['target_class'] = 0
df.loc[df['risk_propensity'] >= q60, 'target_class'] = 1
df.loc[df['risk_propensity'] >= q85, 'target_class'] = 2

print(f"📊 target_class:\n{df['target_class'].value_counts().sort_index()}")

📊 target_class:
target_class
0    4603
1    1764
2    1305
Name: count, dtype: int64


In [7]:
# category_multiplier — множитель по категориям:
# ЖНВЛП × 1.8 (больше продаж).
# Наркотик × 0.15 (меньше — сложный оборот).
# Российский × 1.4 (проще купить).
# С аналогами × 1.2 (больше выбор).
# trend — для класса 2 падение продаж (1.2 → 0.7), для класса 1 слегка падает, для класса 0 стабильно.
# noise = normal(1, 0.15) — ±30% в 95% случаев.
# Сохраняем в drug_sales_2_synthetic.parquet.

# Цель — сгенерировать правдоподобные продажи, чтобы:
# Рассчитать sales_last_month (последний месяц).
# Рассчитать sales_trend (тренд за 2 месяца).
# Рассчитать avg_sales_3m (среднее за 3 месяца).
# Использовать в модели как признаки (X).
# Для 20% строк — использовать в исключениях target (динамика корректирует static_risk).
# Без этих признаков модель не увидит «динамику» — только статику. 
# А с динамикой — увидит исключения (сюрпризы).

N_MONTHS = 24
today = pd.Timestamp.today()
start_date = (today - pd.DateOffset(months=N_MONTHS)).replace(day=1)
months = pd.date_range(start=start_date, periods=N_MONTHS, freq='MS')

seasonality_map = {
    1: 1.2, 2: 1.2, 3: 1.0, 4: 1.0, 5: 1.0, 6: 0.7,
    7: 0.7, 8: 0.7, 9: 1.1, 10: 1.1, 11: 1.1, 12: 1.2,
}

df['category_multiplier'] = 1.0
df.loc[df['is_essential'] == 1, 'category_multiplier'] *= 1.8
df.loc[df['is_narcotic'] == 1, 'category_multiplier'] *= 0.15
df.loc[df['has_russian_producer'] == 1, 'category_multiplier'] *= 1.4
df.loc[df['has_atc_substitutes'] == 1, 'category_multiplier'] *= 1.2

df['base_sales_monthly'] = df['popularity_score'] * df['category_multiplier'] * 100

sales_rows = []
for _, row in df.iterrows():
    smnn = row['smnn_code']
    base = row['base_sales_monthly']
    target = row['target_class']

    if target == 2:
        trend = np.linspace(1.2, 0.7, N_MONTHS)
    elif target == 1:
        trend = np.linspace(1.0, 0.95, N_MONTHS)
    else:
        trend = np.ones(N_MONTHS)

    for i, month in enumerate(months):
        season = seasonality_map[month.month]
        noise = np.random.normal(1.0, 0.15)
        sales = max(1, int(round(base * trend[i] * season * noise)))
        sales_rows.append({
            'smnn_code': smnn,
            'month': month,
            'sales_quantity': sales,
        })

df_sales = pd.DataFrame(sales_rows)
print(f"✅ df_sales: {len(df_sales):,} строк")

# сохраняем 
df_sales.to_parquet(f'{output_dir}/drug_sales_2_synthetic.parquet', index=False)
print(f"✅ Сохранено: drug_sales_2_synthetic.parquet")

✅ df_sales: 184,128 строк
✅ Сохранено: drug_sales_2_synthetic.parquet


In [8]:
# Нормативы запасов: наркотики = 15 дней, ЖНВЛП = 90, обычные = 45.
# Для каждого класса — свой диапазон stock_days:
# класс 2: 3–14 (мало запасов).
# класс 1: 14–30.
# класс 0: 30–90 (много).
# stock = daily_sales * stock_days * noise — шум ±15%.
# Сохраняем в drug_stock_2_synthetic.parquet.

# Этот блок делает 3 вещи:
# Создаёт нормативы запасов (15/45/90 дней) по категориям.
# Генерирует дни запаса по классам (3–14 / 14–30 / 30–90).
# Переводит дни в абсолютные упаковки (daily_sales × days × noise).
# Зачем:
# Правдоподобная синтетика (аптеки не увидят абсурдных чисел).
# Согласовано с риском (высокий риск → мало остатков).
# 20% исключений (для нового target).
# UI показывает «дни запаса».

def get_stock_norm(is_essential, is_narcotic):
    if is_narcotic == 1:
        return 15
    elif is_essential == 1:
        return 90
    else:
        return 45

df['stock_norm_days'] = df.apply(
    lambda r: get_stock_norm(r['is_essential'], r['is_narcotic']), axis=1
)

df_merged = df[['smnn_code', 'target_class', 'stock_norm_days']].merge(
    df_sales, on='smnn_code', how='right'
)

np.random.seed(RANDOM_SEED)
n_rows = len(df_merged)
stock_days_target = np.zeros(n_rows)

for target in [0, 1, 2]:
    mask = df_merged['target_class'] == target
    n = mask.sum()
    if n == 0:
        continue

    if target == 2:
        low, high = 3, 14
        stock_days_target[mask] = np.random.uniform(low, high, n)
    elif target == 1:
        low, high = 14, 30
        stock_days_target[mask] = np.random.uniform(low, high, n)
    else:
        norms = df_merged.loc[mask, 'stock_norm_days'].values
        low = np.maximum(30, norms * 0.7)
        high = np.minimum(norms, 90)
        stock_days_target[mask] = np.random.uniform(low, high)

df_merged['daily_sales'] = df_merged['sales_quantity'] / 30
noise = np.random.normal(1.0, 0.15, n_rows)
df_merged['stock_quantity'] = (
    df_merged['daily_sales'] * stock_days_target * noise
).round().astype(int).clip(lower=1)

df_stock = df_merged[['smnn_code', 'month', 'stock_quantity']].copy()

print(f"✅ df_stock: {len(df_stock):,} строк")

# Сохраняем
df_stock.to_parquet(f'{output_dir}/drug_stock_2_synthetic.parquet', index=False)
print(f"✅ Сохранено: drug_stock_2_synthetic.parquet")

✅ df_stock: 184,128 строк
✅ Сохранено: drug_stock_2_synthetic.parquet


In [9]:
# Из 24 месяцев берём:
# последний месяц → sales_last_month, current_stock.
# предпоследний → для трендов (sales_trend, stock_trend).
# последние 3 → средние (avg_sales_3m, avg_stock_3m).
# current_stock_days = current_stock / (sales_last_month / 30) — прямой прокси target (в X не пойдёт).
# stock_to_sales_ratio — тоже прокси (в X не пойдёт).
# Остальные 6 динамических — в X пойдут (модель увидит).

# Склеивает продажи + остатки → 1 таблица.
# Берёт последний месяц (текущее состояние).
# Берёт предпоследний (для трендов).
# Считает 6 динамических признаков + 2 прямых прокси.
# Сжимает 184,128 строк → 7,672 (1 на СМНН).
# Что пойдёт в модель (X): 6 динамических (без current_stock_days и stock_to_sales_ratio).
# Что останется в dyn_features для UI/исключений: все 9 (включая прямые прокси — для отображения «15 дней запаса» и 20% исключений).

df_dyn = df_sales.merge(df_stock, on=['smnn_code', 'month'], how='inner')
df_dyn = df_dyn.sort_values(['smnn_code', 'month']).reset_index(drop=True)

last_month = df_dyn['month'].max()
last_data = df_dyn[df_dyn['month'] == last_month].copy()

prev_month = sorted(df_dyn['month'].unique())[-2]
prev_data = df_dyn[df_dyn['month'] == prev_month][['smnn_code', 'sales_quantity', 'stock_quantity']]
prev_data = prev_data.rename(columns={
    'sales_quantity': 'sales_prev',
    'stock_quantity': 'stock_prev',
})

dyn_features = last_data[['smnn_code', 'sales_quantity', 'stock_quantity']].copy()
dyn_features = dyn_features.rename(columns={
    'sales_quantity': 'sales_last_month',
    'stock_quantity': 'current_stock',
})
dyn_features = dyn_features.merge(prev_data, on='smnn_code', how='left')

dyn_features['current_stock_days'] = (
    dyn_features['current_stock'] / (dyn_features['sales_last_month'] / 30)
).round(1)

dyn_features['sales_trend'] = (
    dyn_features['sales_last_month'] / dyn_features['sales_prev'].replace(0, np.nan)
).fillna(1.0).round(3)

dyn_features['stock_trend'] = (
    dyn_features['current_stock'] / dyn_features['stock_prev'].replace(0, np.nan)
).fillna(1.0).round(3)

last_3_months = sorted(df_dyn['month'].unique())[-3:]
last_3 = df_dyn[df_dyn['month'].isin(last_3_months)]

avg_sales_3m = last_3.groupby('smnn_code')['sales_quantity'].mean().reset_index()
avg_sales_3m.columns = ['smnn_code', 'avg_sales_3m']
avg_stock_3m = last_3.groupby('smnn_code')['stock_quantity'].mean().reset_index()
avg_stock_3m.columns = ['smnn_code', 'avg_stock_3m']

dyn_features = dyn_features.merge(avg_sales_3m, on='smnn_code', how='left')
dyn_features = dyn_features.merge(avg_stock_3m, on='smnn_code', how='left')

dyn_features['stock_to_sales_ratio'] = (
    dyn_features['current_stock'] / dyn_features['avg_sales_3m'].replace(0, np.nan)
).fillna(0).round(2)

dyn_features = dyn_features.drop(columns=['sales_prev', 'stock_prev'])

print(f"✅ dyn_features: {len(dyn_features):,} строк × {len(dyn_features.columns)} колонок")

✅ dyn_features: 7,672 строк × 9 колонок


In [10]:
# Шаг 1: static_risk — только из статических признаков:
# Веса:
# Нет аналогов (has_atc_substitutes=0) → +3 (критично).
# Мало производителей (<3) → +2.
# Нет российского производителя → +1.
# Наркотик → +1.
# ЖНВЛП → -1 (защита).
# Больше 2 стран → -1 (гео).
# Пороги: ≥5 → высокий, ≥2 → средний, <2 → низкий.
# Шаг 2: 20% исключений:
# np.random.random(n) < 0.20 — маска 20% строк (случайно).
# Для этих 20% — корректируем по динамике:
# static=0 и days<7 → 2 (сюрприз).
# static=2 и days>60 → 1 (сглаживание).
# Для 80% — target = static_risk (без изменений).
# Итого:
# 80% строк → deficit_risk = static_risk (модель учится на статике).
# 20% строк → deficit_risk корректируется по динамике (модель видит исключения).
# Что проверяем в выводе:
# Совпадение static_risk и deficit_risk ≈ 80% (80% не скорректировались).
# Скорректировано примерно 20% (1,534 из 7,672).

# ============================================================
# deficit_risk (static + exceptions) 
# ============================================================
# - static_score: непрерывный (tie-breaking шум ±0.01)
# - static_risk: процентили 60/85 → 60/25/15
# - exceptions: маска 25%, мягкие пороги → ~12-15% изменений
# ============================================================

print("deficit_risk (static + exceptions)")

df_full = df.merge(dyn_features, on='smnn_code', how='inner')
df_full = df_full.reset_index(drop=True)
print(f"\n✅ df_full: {len(df_full):,} строк × {len(df_full.columns)} колонок")


# --- Шаг 1: static_score ---
def calc_static_score_continuous(row):
    score = 0.0
    score += 3.0 * (1 - row['has_atc_substitutes'])
    producers_clip = max(0, min(3, 3 - row['producers_count']))
    score += 2.0 * (producers_clip / 3.0)
    score += 1.0 * (1 - row['has_russian_producer'])
    score += 1.0 * row['is_narcotic']
    score -= 1.0 * row['is_essential']
    countries_clip = max(0, min(3, row['countries_count'] - 2))
    score -= 1.0 * (countries_clip / 3.0)
    return score

df_full['static_score'] = df_full.apply(calc_static_score_continuous, axis=1)

# шум ±0.01
np.random.seed(RANDOM_SEED)
df_full['static_score'] += np.random.uniform(-0.01, 0.01, len(df_full))

print(f"\n📊 static_score:")
print(df_full['static_score'].describe().round(3))
print(f"Уникальных значений: {df_full['static_score'].nunique()}")


# --- static_risk по процентилям ---
q60 = df_full['static_score'].quantile(0.60)
q85 = df_full['static_score'].quantile(0.85)

df_full['static_risk'] = 0
df_full.loc[df_full['static_score'] >= q60, 'static_risk'] = 1
df_full.loc[df_full['static_score'] >= q85, 'static_risk'] = 2

print(f"\n📊 static_risk (ожидание 60/25/15):")
sr_counts = df_full['static_risk'].value_counts().sort_index()
for cls, cnt in sr_counts.items():
    pct = cnt / len(df_full) * 100
    print(f"  Класс {cls}: {cnt:,} ({pct:.1f}%)")


# --- 25% маски + МЯГКИЕ пороги ---
np.random.seed(RANDOM_SEED + 1)
n = len(df_full)
exceptions_mask = np.random.random(n) < 0.50
print(f"\n📊 Маска: {exceptions_mask.sum():,} ({exceptions_mask.mean()*100:.1f}%)")


def apply_dynamic_exception(row, is_exception):
    if not is_exception:
        return row['static_risk']
    
    static = row['static_risk']
    days = row['current_stock_days']
    
    # ============================================================
    # static = 0 (низкий, обычно 30-90 дней)
    # Сюрприз: остатков меньше типичного
    # ============================================================
    if static == 0:
        if days < 15:    # очень мало
            return 2
        if days < 25:    # на грани
            return 1
        if days < 35:    # чуть ниже нормы
            return 1
    
    # ============================================================
    # static = 1 (средний, обычно 14-30 дней)
    # ============================================================
    if static == 1:
        if days < 7:     # критично мало
            return 2
        if days < 12:    # мало
            return 2
        if days > 40:    # много
            return 0
        if days > 55:    # очень много
            return 0
    
    # ============================================================
    # static = 2 (высокий, обычно 3-14 дней)
    # ============================================================
    if static == 2:
        if days > 20:    # больше типичного
            return 1
        if days > 30:    # много
            return 1
        if days > 45:    # очень много
            return 0
    
    return static


df_full['deficit_risk'] = [
    apply_dynamic_exception(row, exc)
    for (_, row), exc in zip(df_full.iterrows(), exceptions_mask)
]


# --- Итог ---
print(f"\n📊 Итоговый deficit_risk:")
risk_counts = df_full['deficit_risk'].value_counts().sort_index()
risk_pct = (risk_counts / len(df_full) * 100).round(2)
for cls, count in risk_counts.items():
    name = {0: 'Низкий', 1: 'Средний', 2: 'Высокий'}[cls]
    print(f"  Класс {cls} ({name}): {count:,} ({risk_pct[cls]}%)")

match_pct = (df_full['static_risk'] == df_full['deficit_risk']).mean() * 100
corrected = (df_full['static_risk'] != df_full['deficit_risk']).sum()

print(f"\n📊 Совпадение: {match_pct:.1f}%   (ожидание: 82–88%)")
print(f"📊 Скорректировано: {corrected:,} ({corrected/len(df_full)*100:.1f}%)   (ожидание: 12–18%)")

deficit_risk (static + exceptions)

✅ df_full: 7,672 строк × 49 колонок

📊 static_score:
count    7672.000
mean        0.907
std         1.642
min        -2.010
25%        -0.331
50%         0.671
75%         1.993
max         5.343
Name: static_score, dtype: float64
Уникальных значений: 7672

📊 static_risk (ожидание 60/25/15):
  Класс 0: 4,603 (60.0%)
  Класс 1: 1,918 (25.0%)
  Класс 2: 1,151 (15.0%)

📊 Маска: 3,723 (48.5%)

📊 Итоговый deficit_risk:
  Класс 0 (Низкий): 4,304 (56.1%)
  Класс 1 (Средний): 2,108 (27.48%)
  Класс 2 (Высокий): 1,260 (16.42%)

📊 Совпадение: 88.9%   (ожидание: 82–88%)
📊 Скорректировано: 854 (11.1%)   (ожидание: 12–18%)


In [11]:
# ============================================================
# Финальный df_ml_2 
# ============================================================

COLS_TO_EXCLUDE = [
    # Прямые прокси target (утечка)
    'current_stock_days',
    'stock_to_sales_ratio',

    # Промежуточные / технические
    'popularity_score',
    'risk_propensity',
    'target_class',
    'static_risk',
    'static_score',
    'stock_norm_days',
    'category_multiplier',
    'base_sales_monthly',

    # Высококардинальные категории (шум для модели)
    'atc_level4_mode',
    'atc_class_mode',
]

cols_final = [c for c in df_full.columns if c not in COLS_TO_EXCLUDE]
df_ml_2 = df_full[cols_final].copy()

print(f"✅ df_ml_2: {len(df_ml_2):,} строк × {len(df_ml_2.columns)} колонок")
print(f"📋 Колонки: {list(df_ml_2.columns)}")

df_ml_2.to_parquet(f'{output_dir}/df_ml_2.parquet', index=False)
print(f"✅ Сохранено: df_ml_2.parquet")


# ============================================================
# Проверки реализма
# ============================================================
print("ПРОВЕРКИ РЕАЛИЗМА")

print(f"\n📊 Средние stock_days (is_essential):")
ess_days = df_full.groupby('is_essential')['current_stock_days'].mean()
print(f"True (ЖНВЛП): {ess_days.get(True, 0):.1f} дней")
print(f"False (не-ЖНВЛП): {ess_days.get(False, 0):.1f} дней")
print(f"{'✅' if ess_days.get(True, 0) > ess_days.get(False, 0) else '⚠️'} ЖНВЛП > не-ЖНВЛП")

print(f"\n📊 Средние stock_days (is_narcotic):")
narc_days = df_full.groupby('is_narcotic')['current_stock_days'].mean()
print(f"True (наркотики): {narc_days.get(True, 0):.1f} дней")
print(f"False (не-наркотики): {narc_days.get(False, 0):.1f} дней")
print(f"{'✅' if narc_days.get(True, 0) < narc_days.get(False, 0) else '⚠️'} Наркотики < не-наркотики")

print(f"\n📊 Средний deficit_risk (has_atc_substitutes):")
atc_risk = df_full.groupby('has_atc_substitutes')['deficit_risk'].mean()
print(f"0 (нет аналогов): {atc_risk.get(0, 0):.2f}")
print(f"1 (есть аналоги): {atc_risk.get(1, 0):.2f}")
print(f"{'✅' if atc_risk.get(0, 0) > atc_risk.get(1, 0) else '⚠️'} Нет аналогов > есть аналоги")

print(f"\n📊 Средний deficit_risk (producers_count == 1):")
prod_risk = df_full.groupby(df_full['producers_count'] == 1)['deficit_risk'].mean()
print(f"True (1 производитель): {prod_risk.get(True, 0):.2f}")
print(f"False (2+): {prod_risk.get(False, 0):.2f}")
print(f"{'✅' if prod_risk.get(True, 0) > prod_risk.get(False, 0) else '⚠️'} 1 производитель > 2+")


# ============================================================
# Финальная проверка df_ml_2
# ============================================================
print("ФИНАЛЬНЫЕ ПРОВЕРКИ df_ml_2")
print(f"\n📊 NaN в df_ml_2: {df_ml_2.isnull().sum().sum()}")
print(f" {'✅' if df_ml_2.isnull().sum().sum() == 0 else '⚠️'} нет NaN")

assert 'current_stock_days' not in df_ml_2.columns, "❌ current_stock_days остался!"
assert 'stock_to_sales_ratio' not in df_ml_2.columns, "❌ stock_to_sales_ratio остался!"
assert 'static_risk' not in df_ml_2.columns, "❌ static_risk остался!"
assert 'static_score' not in df_ml_2.columns, "❌ static_score остался!"
print(f" ✅ Прямых прокси и промежуточных колонок нет")

print(f"\n📊 deficit_risk:")
print(df_ml_2['deficit_risk'].value_counts().sort_index())

✅ df_ml_2: 7,672 строк × 40 колонок
📋 Колонки: ['smnn_code', 'standard_inn', 'products_count', 'producers_count', 'trade_count', 'form_count', 'doze_count', 'reg_num_count', 'atc_group_mode', 'has_combined_atc', 'is_essential', 'is_narcotic', 'country_group_mode', 'has_russian_producer', 'countries_count', 'primary_unit_type', 'primary_unit_name', 'avg_age', 'min_age', 'max_age', 'log_products_count', 'log_klp_code_count', 'log_reg_num_count', 'log_trade_count', 'log_producers_count', 'products_per_producer', 'log_products_per_producer', 'reg_num_per_producer', 'trade_per_producer', 'atc_is_unknown', 'atc_substitutes_count', 'has_atc_substitutes', 'log_atc_substitutes_count', 'sales_last_month', 'current_stock', 'sales_trend', 'stock_trend', 'avg_sales_3m', 'avg_stock_3m', 'deficit_risk']
✅ Сохранено: df_ml_2.parquet
ПРОВЕРКИ РЕАЛИЗМА

📊 Средние stock_days (is_essential):
True (ЖНВЛП): 75.2 дней
False (не-ЖНВЛП): 24.3 дней
✅ ЖНВЛП > не-ЖНВЛП

📊 Средние stock_days (is_narcotic):
True (н

# Загрузка синтетики в БД  
(Раскомментировать в случае необходимости загрузки новых данных)

In [ ]:
# # Создаём таблицы:
# #   - synthetic_data.drug_sales_2    (продажи)
# #   - synthetic_data.drug_stock_2    (остатки)
# #   - synthetic_data.drug_features_2 (ML-датасет — все 38 колонок)
# # ============================================================
# # --- Удаляем старые _2, если были ---
# with engine.begin() as conn:
#     conn.execute(text("DROP TABLE IF EXISTS synthetic_data.drug_sales_2 CASCADE"))
#     conn.execute(text("DROP TABLE IF EXISTS synthetic_data.drug_stock_2 CASCADE"))
#     conn.execute(text("DROP TABLE IF EXISTS synthetic_data.drug_features_2 CASCADE"))

# print("✅ Старые таблицы удалены (если были)")

# # ============================================================
# # 1. drug_sales_2
# # ============================================================
# df_sales = pd.read_parquet(f'{output_dir}/drug_sales_2_synthetic.parquet')
# df_sales['month'] = pd.to_datetime(df_sales['month']).dt.date
# df_sales['sales_quantity'] = df_sales['sales_quantity'].astype(int)

# df_sales.to_sql(
#     'drug_sales_2', engine, schema='synthetic_data',
#     if_exists='replace', index=False, method='multi', chunksize=10000,
# )
# print(f"✅ drug_sales_2: {len(df_sales):,} строк")

# # ============================================================
# # 2. drug_stock_2
# # ============================================================
# df_stock = pd.read_parquet(f'{output_dir}/drug_stock_2_synthetic.parquet')
# df_stock['month'] = pd.to_datetime(df_stock['month']).dt.date
# df_stock['stock_quantity'] = df_stock['stock_quantity'].astype(int)

# df_stock.to_sql(
#     'drug_stock_2', engine, schema='synthetic_data',
#     if_exists='replace', index=False, method='multi', chunksize=10000,
# )
# print(f"✅ drug_stock_2: {len(df_stock):,} строк")

# # ============================================================
# # 3. drug_features_2 — ВСЕ колонки из df_ml_2
# # ============================================================
# df_ml_2 = pd.read_parquet(f'{output_dir}/df_ml_2.parquet')

# # Приводим типы (bool → int, чтобы избежать проблем с SQLAlchemy)
# df_ml_2_load = df_ml_2.copy()
# for col in df_ml_2_load.columns:
#     if df_ml_2_load[col].dtype == bool:
#         df_ml_2_load[col] = df_ml_2_load[col].astype(int)

# df_ml_2_load.to_sql(
#     'drug_features_2', engine, schema='synthetic_data',
#     if_exists='replace', index=False, method='multi', chunksize=2000,
# )
# print(f"✅ drug_features_2: {len(df_ml_2_load):,} строк × {len(df_ml_2_load.columns)} колонок")

# # ============================================================
# # Проверки
# # ============================================================
# with engine.connect() as conn:
#     for table in ['drug_sales_2', 'drug_stock_2', 'drug_features_2']:
#         cnt = conn.execute(text(f"SELECT COUNT(*) FROM synthetic_data.{table}")).scalar()
#         print(f"synthetic_data.{table}: {cnt:,} строк")

#     print(f"\n📊 Колонки в drug_features_2:")
#     cols = conn.execute(text("""
#         SELECT column_name, data_type
#         FROM information_schema.columns
#         WHERE table_schema='synthetic_data' AND table_name='drug_features_2'
#         ORDER BY ordinal_position
#     """)).fetchall()
#     for name, dtype in cols:
#         print(f"   {name}: {dtype}")

#     print(f"\n📊 Распределение deficit_risk в drug_features_2:")
#     result = conn.execute(text("""
#         SELECT deficit_risk, COUNT(*)
#         FROM synthetic_data.drug_features_2
#         GROUP BY deficit_risk
#         ORDER BY deficit_risk
#     """)).fetchall()
#     for risk, cnt in result:
#         pct = cnt / sum(r[1] for r in result) * 100
#         print(f"   Класс {risk}: {cnt:,} ({pct:.1f}%)")